# Evaluators

从总体上看，评估器会根据参考示例来判断您的 LLM 应用程序的调用情况，并返回一个评估分数。

在 LangSmith 评估器中，我们将此过程表示为一个函数，该函数接受一个 Run（表示 LLM 应用程序调用）和一个 Example（表示要评估的数据点），并返回 Feedback（表示评估器对 LLM 应用程序调用的评分）。

![Evaluator](../../images/evaluator.png)

以下是一个非常简单的自定义评估器示例，它将模型的输出与数据集中的预期输出进行比较：

In [1]:
from langsmith.schemas import Example, Run

def correct_label(inputs: dict, reference_outputs: dict, outputs: dict) -> dict:
  score = outputs.get("output") == reference_outputs.get("label")
  return {"score": int(score), "key": "correct_label"}

### LLM-as-Judge Evaluation

LLM 作为评判标准，评估器使用 LLM 对系统输出进行评分。使用时，通常需要在 LLM 提示中编码评分规则/标准。LLM 可以不依赖参考资料（例如，检查系统输出是否包含冒犯性内容或是否符合特定标准）。或者，LLM 可以将任务输出与参考资料进行比较（例如，检查输出相对于参考资料是否准确）。

以下是一个如何定义具有结构化输出的 LLM 作为评判者的评估示例

In [ ]:
# You can set them inline
# import os
# os.environ["OPENAI_API_KEY"] = ""

In [ ]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

In [8]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

class Similarity_Score(BaseModel):
    similarity_score: int = Field(description="Semantic similarity score between 1 and 10, where 1 means unrelated and 10 means identical.")

# NOTE: This is our evaluator
def compare_semantic_similarity(inputs: dict, reference_outputs: dict, outputs: dict):
    input_question = inputs["question"]
    reference_response = reference_outputs["output"]
    run_response = outputs["output"]
    
    completion = client.beta.chat.completions.parse(
        model="qwen3-max",
        messages=[
            {   
                "role": "system",
                "content": (
                    "You are a semantic similarity evaluator. Compare the meanings of two responses to a question, "
                    "Reference Response and New Response, where the reference is the correct answer, and we are trying to judge if the new response is similar. "
                    "Provide a score between 1 and 10, where 1 means completely unrelated, and 10 means identical in meaning. "
                    "Output your response in JSON format with the key 'similarity_score'."
                ),
                # "Output your response in JSON format with the key 'similarity_score'."
                # DashScope的兼容模式API有特殊要求：当请求JSON格式响应时，提示中必须显式包含"json"这个词。
                # 这是为了确保模型理解需要结构化输出。虽然OpenAI原生API不需要这个要求，但DashScope的兼容层添加了这个验证规则。
            },
            {"role": "user", "content": f"Question: {input_question}\n Reference Response: {reference_response}\n Run Response: {run_response}"}
        ],
        response_format=Similarity_Score,
    )

    similarity_score = completion.choices[0].message.parsed
    return {"score": similarity_score.similarity_score, "key": "similarity"}


<div class="alert alert-block alert-danger">
    <b>注意：</b> 我们故意给出错误答案，因此我们预期会得到低分。
</div>

In [9]:
# From Dataset Example
inputs = {
  "question": "Is LangSmith natively integrated with LangChain?"
}
reference_outputs = {
  "output": "Yes, LangSmith is natively integrated with LangChain, as well as LangGraph."
}


# From Run
outputs = {
  "output": "No, LangSmith is NOT integrated with LangChain."
}

similarity_score = compare_semantic_similarity(inputs, reference_outputs, outputs)
print(f"Semantic similarity score: {similarity_score}")

Semantic similarity score: {'score': 1, 'key': 'similarity'}


还可以直接使用 `Run` 和 `Example` 定义评估器！这种方式比较旧，实际上还是推荐使用上面的方式。

In [10]:
from langsmith.schemas import Run, Example

def compare_semantic_similarity_v2(root_run: Run, example: Example):
    input_question = example["inputs"]["question"]
    reference_response = example["outputs"]["output"]
    run_response = root_run["outputs"]["output"]
    
    completion = client.beta.chat.completions.parse(
        model="qwen3-max",
        messages=[
            {   
                "role": "system",
                "content": (
                    "You are a semantic similarity evaluator. Compare the meanings of two responses to a question, "
                    "Reference Response and New Response, where the reference is the correct answer, and we are trying to judge if the new response is similar. "
                    "Provide a score between 1 and 10, where 1 means completely unrelated, and 10 means identical in meaning. "
                    "Output your response in JSON format with the key 'similarity_score'."
                ),
            },
            {"role": "user", "content": f"Question: {input_question}\n Reference Response: {reference_response}\n Run Response: {run_response}"}
        ],
        response_format=Similarity_Score,
    )

    similarity_score = completion.choices[0].message.parsed
    return {"score": similarity_score.similarity_score, "key": "similarity"}

In [11]:
sample_run = {
  "name": "Sample Run",
  "inputs": {
    "question": "Is LangSmith natively integrated with LangChain?"
  },
  "outputs": {
    "output": "No, LangSmith is NOT integrated with LangChain."
  },
  "is_root": True,
  "status": "success",
  "extra": {
    "metadata": {
      "key": "value"
    }
  }
}

sample_example = {
  "inputs": {
    "question": "Is LangSmith natively integrated with LangChain?"
  },
  "outputs": {
    "output": "Yes, LangSmith is natively integrated with LangChain, as well as LangGraph."
  },
  "metadata": {
    "dataset_split": [
      "AI generated",
      "base"
    ]
  }
}

similarity_score = compare_semantic_similarity_v2(sample_run, sample_example)
print(f"Semantic similarity score: {similarity_score}")

Semantic similarity score: {'score': 1, 'key': 'similarity'}
